# Cross-Modal Diagnostic Observability — Stage 11F-R

## Development-only directed edge assembly and DDO-2 fit-readiness audit

**Run only after Stage 11E-R completed with handoff hash** `98a285c18d7e680a126b78b5dfa70a4883e7c8cf9536aff2a76528adb42f2a5d`.

This notebook:

1. verifies the sealed Stage 11E-R source axes, nine-edge roster, Stage 8B 21-edge library, and Stage 9 unfitted DDO-2 specification;
2. applies each frozen recoverable ultrasound source axis to the other released held-out ultrasound domains;
3. freezes label-free components and target probabilities before this stage joins target labels;
4. evaluates all nine pre-authorised development edges once;
5. appends them to the historical 21-edge library only after lineage, completeness, and duplication checks;
6. rebuilds leave-one-dataset-out and leave-one-modality-out folds and applies the **Stage 9 frozen fit gates**; and
7. authorises Stage 11G-R development-only fitting only if every assembly and fit-readiness gate passes.

### Interpretation boundary

The released ultrasound held-out labels were already observed in Stage 11E-R for source-recoverability assessment. The Stage 11F-R freeze is therefore a **mechanical development label-isolation boundary**, not a new prospective blind unseal. It proves that Stage 11F-R label-free construction did not join target outcomes before its prediction freeze.

### Still prohibited

- DDO-2 coefficient fitting in this notebook;
- source-axis refitting or failed-source rescue;
- outcome-driven feature replacement, threshold tuning, or edge deletion;
- Stage 12 construction or execution; and
- any locked-blind asset access.

CPU runtime is sufficient. Run all seven code cells from top to bottom.

In [10]:
# @title 11F-R-0. Mount Drive, verify sealed parents, and freeze the assembly protocol
import hashlib, json, os, platform, sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

DEFAULT_ROOT = Path('/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability') if IN_COLAB else Path('/tmp/Cross-Modal_Diagnostic_Observability')
PROJECT_ROOT = Path(os.environ.get('CDO_PROJECT_ROOT', str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT/'05_Code'/'Cross_Modal'
CM_ROOT = PROJECT_ROOT/'06_Data_Records'/'Cross_Modal'

STAGE11D_ROOT = CM_ROOT/'Stage11D-R_Cross_Roster_Dedup_Exact_Manifest_And_Grouped_Split_Freeze_v0.1'
STAGE11E_ROOT = CM_ROOT/'Stage11E-R_Development_Only_Source_Recoverability_And_Axis_Freeze_v0.1'
STAGE8B_ROOT = CM_ROOT/'Stage8B_NLM_Chest_Radiography_Access_Completion_v0.1'
STAGE9_ROOT = CM_ROOT/'Stage9_Hierarchical_DDO2_Specification_Freeze_v0.1'
ROOT = CM_ROOT/'Stage11F-R_Development_Only_Directed_Edge_Assembly_And_Fit_Readiness_v0.2'

P0,P1,P2,P3,P4,P5,P6 = [ROOT/x for x in [
    '00_Protocol','01_LabelFree_Prediction_Freeze','02_Development_Transfer_Outcomes',
    '03_Expanded_Edge_Library','04_Independence_And_Readiness','05_Firewall','06_Results'
]]
for path in [CODE_ROOT,P0,P1,P2,P3,P4,P5,P6]:
    path.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = 'CrossModal_Stage11F-R_Development_Only_Directed_Edge_Assembly_And_Fit_Readiness_v0.2.ipynb'
NOTEBOOK_PATH = CODE_ROOT/NOTEBOOK_NAME

PARENT_SPLITS = STAGE11D_ROOT/'03_Grouped_Splits'/'Stage11D-R_Frozen_Grouped_Split_Manifest_v0.1.csv'
PARENT_PROTOCOL = STAGE11E_ROOT/'00_Protocol'/'Stage11E-R_Protocol_Seal_v0.1.json'
PARENT_EMBEDDING_MANIFEST = STAGE11E_ROOT/'01_Frozen_Embeddings'/'Stage11E-R_Frozen_Embedding_Manifest_v0.1.csv'
PARENT_OOF_PREDICTIONS = STAGE11E_ROOT/'02_Development_OOF'/'Stage11E-R_Development_OOF_Predictions_v0.1.csv'
PARENT_AXIS_MANIFEST = STAGE11E_ROOT/'03_Frozen_Source_Axes'/'Stage11E-R_Frozen_Source_Axis_Manifest_v0.1.csv'
PARENT_AXIS_FREEZE = STAGE11E_ROOT/'03_Frozen_Source_Axes'/'Stage11E-R_Prevalidation_Axis_Freeze_v0.1.json'
PARENT_HELDOUT_PREDICTIONS = STAGE11E_ROOT/'04_Heldout_Validation'/'Stage11E-R_Heldout_Predictions_v0.1.csv'
PARENT_RECOVERABILITY = STAGE11E_ROOT/'04_Heldout_Validation'/'Stage11E-R_Source_Recoverability_Decisions_v0.1.csv'
PARENT_EDGE_ROSTER = STAGE11E_ROOT/'04_Heldout_Validation'/'Stage11E-R_Authorised_Development_Edge_Roster_v0.1.csv'
PARENT_HANDOFF = STAGE11E_ROOT/'04_Heldout_Validation'/'Stage11E-R_Stage11F-R_Handoff_v0.1.json'
PARENT_FINAL = STAGE11E_ROOT/'06_Results'/'Stage11E-R_Complete_v0.1.json'

STAGE8B_LIBRARY = STAGE8B_ROOT/'05_Unsealed_Chest_Discovery'/'Stage8B_ThreeModality_Eligible_Edge_Library_v0.1.csv'
STAGE8B_FINAL = STAGE8B_ROOT/'06_Results'/'Stage8B_NLM_Access_Completion_Complete_v0.1.json'
STAGE9_PROTOCOL = STAGE9_ROOT/'00_Protocol'/'Stage9_Hierarchical_DDO2_Specification_Protocol_Seal_v0.1.json'
STAGE9_MODEL_SPEC = STAGE9_ROOT/'02_Frozen_Specification'/'Stage9_Frozen_Hierarchical_DDO2_Model_Specification_v0.1.json'
STAGE9_FIT_AUDIT = STAGE9_ROOT/'03_Grouped_Validation_Protocol'/'Stage9_DDO2_Coefficient_Fit_Eligibility_Audit_v0.1.csv'
STAGE9_FINAL = STAGE9_ROOT/'05_Results'/'Stage9_Hierarchical_DDO2_Specification_Freeze_Complete_v0.1.json'

EXPECTED_STAGE11E_PROTOCOL = '4362d6a8baed676ee6245971dac40aeb8d2865541df05446ce970ed4431fba72'
EXPECTED_STAGE11E_EMBEDDING_MANIFEST = 'a273cdad4892d8b954dad6dfb2c230e7352f68899897466a95852e2c5baa16e2'
EXPECTED_STAGE11E_AXIS_FREEZE = '98903477c84b7ec6b33258e30f02e8f357ec36e4261c6cbd242fb73ba8c1ec20'
EXPECTED_STAGE11E_RECOVERABILITY = 'a7ad22dcc25e5bef7763246d79fad64536fcbb87eef4493037712da8f07199be'
EXPECTED_STAGE11E_HANDOFF = '98a285c18d7e680a126b78b5dfa70a4883e7c8cf9536aff2a76528adb42f2a5d'
EXPECTED_STAGE11E_FINAL = '24d41b7a3a30b1548a81ab1a99d909eb29bf9189fef49709754d1b9cfc8dcca2'
EXPECTED_STAGE8B_LIBRARY = '60ee87ec253c34d2e3b50f6f1a40318aa1a14c0f0e5bf168fcb60d5f0d641ebf'
EXPECTED_STAGE8B_FINAL = '4339402d843af177ef1181df1e3ae4fb7b0b6bd0211e250bb22cd9b255141024'
EXPECTED_STAGE9_PROTOCOL = 'f5689995ec6ab0dcf0bcacf813d578edd47edbedf8fb96b7809ec674e3cad593'
EXPECTED_STAGE9_MODEL_SPEC = 'e2a0d8f65143e0b9d58d740595a39b803f1684481c4f8af01f05935bfaacb732'
EXPECTED_STAGE9_FINAL = '2568a9fdaff83ab74938655d65d25cd9d50f8b28bafbc44e856098932f6c6648'

RANDOM_SEED = 20260721
N_BOOTSTRAP = 2000
FIXED_THRESHOLD = 0.5
AUC_MINIMUM = 0.70
AUC_CI_LOWER_STRICT_MINIMUM = 0.55
CALIBRATION_DEGRADATION_TOLERANCE = 0.05
OPERATING_POINT_BALANCED_ACCURACY_MINIMUM = 0.70
AXIS_EQUIVALENCE_TOLERANCE = 1e-8
EXPECTED_NEW_EDGES = 9
EXPECTED_PARENT_EDGES = 21
EXPECTED_TOTAL_IF_COMPLETE = 30
MODALITY = 'breast_ultrasound'
TASK = 'malignant_vs_benign_breast_lesion'
LOCKED_ASSET_MARKERS = ['BUSI_CAIRO_2019','OASBUD_2017','DERM7PT_2019']

PROTOCOL = P0/'Stage11F-R_Protocol_Seal_v0.2.json'
PARENT_COMMIT = P0/'Stage11F-R_Parent_Input_Commitment_v0.2.csv'
ENVIRONMENT = P0/'Stage11F-R_Execution_Environment_v0.2.json'
LABEL_FREE_COMPONENTS = P1/'Stage11F-R_LabelFree_Ultrasound_Edge_Components_v0.2.csv'
FROZEN_PREDICTIONS = P1/'Stage11F-R_LabelFree_Ultrasound_Target_Predictions_v0.2.csv'
PREDICTION_FREEZE = P1/'Stage11F-R_PreOutcome_Prediction_Freeze_v0.2.json'
EDGE_MATRIX = P2/'Stage11F-R_Ultrasound_Development_Edge_Matrix_v0.2.csv'
EVALUATED_PREDICTIONS = P2/'Stage11F-R_Evaluated_Ultrasound_Target_Predictions_v0.2.csv'
OUTCOME_SUMMARY = P2/'Stage11F-R_Development_Transfer_Outcome_Summary_v0.2.csv'
EXPANDED_LIBRARY = P3/'Stage11F-R_FourModality_Eligible_Edge_Library_v0.2.csv'
LIBRARY_FREEZE = P3/'Stage11F-R_Expanded_Library_Freeze_v0.2.json'
DUPLICATE_AUDIT = P4/'Stage11F-R_Edge_Duplicate_And_Overlap_Audit_v0.2.csv'
DYNAMIC_RANGE_AUDIT = P4/'Stage11F-R_Outcome_And_Component_Dynamic_Range_Audit_v0.2.csv'
DATASET_INCIDENCE = P4/'Stage11F-R_Dataset_Edge_Incidence_Audit_v0.2.csv'
FOLD_REGISTRY = P4/'Stage11F-R_Frozen_Grouped_Validation_Fold_Registry_v0.2.csv'
FIT_ELIGIBILITY = P4/'Stage11F-R_DDO2_Fit_Eligibility_Audit_v0.2.csv'
ASSEMBLY_DECISION = P4/'Stage11F-R_Assembly_And_Readiness_Decision_v0.2.json'
FIREWALL = P5/'Stage11F-R_Independent_Validity_And_Firewall_Checks_v0.2.csv'
REPORT = P6/'Stage11F-R_Directed_Edge_Assembly_And_Fit_Readiness_Report_v0.2.md'
OUTPUT_MANIFEST = P6/'Stage11F-R_Output_Integrity_Manifest_v0.2.csv'
FINAL = P6/'Stage11F-R_Complete_v0.2.json'
RUNTIME = P6/'Stage11F-R_Runtime_State_v0.2.json'

def now():
    return datetime.now(timezone.utc).isoformat()

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def sha_json(value):
    raw = json.dumps(value, sort_keys=True, separators=(',',':'), ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()

def canonical_csv(frame):
    return frame.fillna('').to_csv(index=False, lineterminator='\n', float_format='%.12g')

def write_text_once(path, text):
    path = Path(path)
    if path.exists():
        assert path.read_text(encoding='utf-8') == text, f'Replay mismatch: {path}'
    else:
        path.write_text(text, encoding='utf-8')

def write_csv_once(path, frame):
    write_text_once(path, canonical_csv(frame))

def write_json_once(path, value):
    write_text_once(path, json.dumps(value, indent=2, ensure_ascii=False)+'\n')

def verify_self(path, hash_field, expected=None):
    value = json.loads(Path(path).read_text(encoding='utf-8'))
    claim = value[hash_field]
    payload = dict(value); payload.pop(hash_field)
    assert sha_json(payload) == claim, f'Self-hash mismatch: {path}'
    if expected is not None:
        assert claim == expected, f'Unexpected sealed hash: {path}'
    return value

def create_or_verify_seal(path, payload, hash_field, time_field):
    path = Path(path)
    if path.exists():
        value = verify_self(path, hash_field)
        for key, expected in payload.items():
            assert value[key] == expected, f'Sealed field changed: {path} :: {key}'
        return value
    value = dict(payload)
    value[time_field] = now()
    value[hash_field] = sha_json(value)
    write_json_once(path, value)
    return value

def notebook_source_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding='utf-8'))
    payload = []
    for cell in notebook.get('cells',[]):
        if cell.get('cell_type') not in {'code','markdown'}:
            continue
        source = cell.get('source',[])
        source = ''.join(source) if isinstance(source,list) else str(source)
        payload.append({'cell_type':cell['cell_type'],'source':source.replace('\r\n','\n')})
    return sha_json(payload)

def markdown_table(frame):
    x = frame.fillna('').astype(str)
    esc = lambda s:str(s).replace('|','\\|').replace('\n',' ')
    return '\n'.join(
        ['| '+' | '.join(esc(c) for c in x.columns)+' |','| '+' | '.join('---' for _ in x.columns)+' |']+
        ['| '+' | '.join(esc(v) for v in row)+' |' for row in x.itertuples(index=False,name=None)]
    )

required = [NOTEBOOK_PATH,PARENT_SPLITS,PARENT_PROTOCOL,PARENT_EMBEDDING_MANIFEST,PARENT_OOF_PREDICTIONS,
            PARENT_AXIS_MANIFEST,PARENT_AXIS_FREEZE,PARENT_HELDOUT_PREDICTIONS,PARENT_RECOVERABILITY,
            PARENT_EDGE_ROSTER,PARENT_HANDOFF,PARENT_FINAL,STAGE8B_LIBRARY,STAGE8B_FINAL,STAGE9_PROTOCOL,
            STAGE9_MODEL_SPEC,STAGE9_FIT_AUDIT,STAGE9_FINAL]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Missing sealed inputs:\n'+'\n'.join(missing)

parent_protocol = verify_self(PARENT_PROTOCOL,'seal_sha256',EXPECTED_STAGE11E_PROTOCOL)
axis_freeze = verify_self(PARENT_AXIS_FREEZE,'freeze_sha256',EXPECTED_STAGE11E_AXIS_FREEZE)
parent_handoff = verify_self(PARENT_HANDOFF,'handoff_sha256',EXPECTED_STAGE11E_HANDOFF)
parent_final = verify_self(PARENT_FINAL,'final_record_sha256',EXPECTED_STAGE11E_FINAL)
stage8b_final = verify_self(STAGE8B_FINAL,'final_record_sha256',EXPECTED_STAGE8B_FINAL)
stage9_protocol = verify_self(STAGE9_PROTOCOL,'seal_sha256',EXPECTED_STAGE9_PROTOCOL)
stage9_model_spec = verify_self(STAGE9_MODEL_SPEC,'specification_sha256',EXPECTED_STAGE9_MODEL_SPEC)
stage9_final = verify_self(STAGE9_FINAL,'final_record_sha256',EXPECTED_STAGE9_FINAL)

assert sha_file(PARENT_EMBEDDING_MANIFEST) == EXPECTED_STAGE11E_EMBEDDING_MANIFEST == parent_handoff['embedding_manifest_sha256']
assert sha_file(PARENT_RECOVERABILITY) == EXPECTED_STAGE11E_RECOVERABILITY == parent_handoff['source_recoverability_summary_sha256']
assert axis_freeze['freeze_sha256'] == parent_handoff['prevalidation_axis_freeze_sha256']
assert parent_handoff['global_source_gate_passed'] is True and parent_handoff['authorised_development_edge_count'] == EXPECTED_NEW_EDGES
assert parent_handoff['stage11f_r_development_only_authorised'] is True
assert parent_handoff['stage12_authorised'] is False and parent_handoff['ddo2_fitted'] is False
assert parent_final['locked_blind_assets_touched'] is False
assert sha_file(STAGE8B_LIBRARY) == EXPECTED_STAGE8B_LIBRARY == stage9_final['edge_library_sha256']
assert stage9_final['model_specification_sha256'] == EXPECTED_STAGE9_MODEL_SPEC
assert stage9_final['fit_authorised'] is False and stage9_final['final_ddo2_fitted'] is False

edge_roster = pd.read_csv(PARENT_EDGE_ROSTER)
recoverability = pd.read_csv(PARENT_RECOVERABILITY)
axis_manifest = pd.read_csv(PARENT_AXIS_MANIFEST)
embedding_manifest = pd.read_csv(PARENT_EMBEDDING_MANIFEST)
historical_library = pd.read_csv(STAGE8B_LIBRARY)
assert len(edge_roster) == EXPECTED_NEW_EDGES and edge_roster.edge_id.nunique() == EXPECTED_NEW_EDGES
assert edge_roster.development_only.fillna(False).astype(bool).all()
assert len(historical_library) == EXPECTED_PARENT_EDGES and historical_library.edge_id.nunique() == EXPECTED_PARENT_EDGES

PASSING_SOURCES = sorted(parent_handoff['recoverable_source_dataset_ids'])
ALL_ULTRASOUND_DATASETS = sorted(parent_handoff['evaluated_source_dataset_ids'])
assert len(PASSING_SOURCES) == 3 and len(ALL_ULTRASOUND_DATASETS) == 4
assert set(edge_roster.source_dataset_id) == set(PASSING_SOURCES)
assert set(edge_roster.target_dataset_id) == set(ALL_ULTRASOUND_DATASETS)

frozen_features = sorted({feature for features in stage9_model_spec['axis_features'].values() for feature in features})
expected_frozen_features = sorted(['support_fraction','fraction_beyond_source_q99','atc_estimated_accuracy',
    'unlabeled_mixture_prevalence','mean_entropy_nats','target_to_source_logit_iqr_ratio'])
assert frozen_features == expected_frozen_features
fit_gates = stage9_model_spec['fit_gates']
assert fit_gates['minimum_training_edges_per_grouped_fold'] == 18

commit = pd.DataFrame([{
    'role':path.name,'relative_path':str(path.relative_to(PROJECT_ROOT)),
    'size_bytes':path.stat().st_size,'sha256':sha_file(path)
} for path in required[1:]])
write_csv_once(PARENT_COMMIT,commit)

protocol_payload = {
    'stage':'Stage11F-R','version':'0.2','scope':'DEVELOPMENT_ONLY_DIRECTED_EDGE_ASSEMBLY_AND_FIT_READINESS',
    'repair_class':'IMPLEMENTATION_ONLY_HISTORICAL_ROW_COMPARISON_REPAIR',
    'supersedes_aborted_version':'0.1',
    'aborted_v01_protocol_seal_sha256':'a292f9270c038eddbea94dbd99609c6e360858890c844ea7082a6824711ed0f6',
    'scientific_contract_changed':False,
    'historical_row_identity_rule':'same columns and edge order; nonnumeric exact; numeric rtol=1e-12 and atol=0',
    'notebook_source_sha256':notebook_source_sha256(NOTEBOOK_PATH),
    'parent_stage11e_r_final_sha256':parent_final['final_record_sha256'],
    'parent_stage11e_r_handoff_sha256':parent_handoff['handoff_sha256'],
    'parent_stage8b_library_sha256':sha_file(STAGE8B_LIBRARY),
    'parent_stage9_final_sha256':stage9_final['final_record_sha256'],
    'parent_stage9_model_specification_sha256':stage9_model_spec['specification_sha256'],
    'authorised_edge_ids':sorted(edge_roster.edge_id.astype(str).tolist()),
    'recoverable_source_dataset_ids':PASSING_SOURCES,
    'retired_source_dataset_ids':sorted(parent_handoff['retired_source_dataset_ids']),
    'target_dataset_ids':ALL_ULTRASOUND_DATASETS,
    'frozen_model_features':frozen_features,
    'stage9_fit_gates':fit_gates,
    'metric_contract':{
        'target_auc':'image_level_roc_auc','target_auc_uncertainty':'released_group_cluster_bootstrap',
        'bootstrap_replicates':N_BOOTSTRAP,'fixed_threshold':FIXED_THRESHOLD,
        'discrimination_pass':f'AUC>={AUC_MINIMUM} and lower95CI>{AUC_CI_LOWER_STRICT_MINIMUM}',
        'calibration_pass':f'ECE and Brier degradation each <= {CALIBRATION_DEGRADATION_TOLERANCE}',
        'operating_point_pass':f'balanced_accuracy_at_0.5 >= {OPERATING_POINT_BALANCED_ACCURACY_MINIMUM}'
    },
    'target_labels_previously_observed_in_parent_source_recoverability':True,
    'stage11f_label_free_target_prediction_files_must_exclude_labels':True,
    'stage11f_target_outcome_join_permitted_only_after_prediction_freeze':True,
    'dynamic_range_role':'descriptive plus non-degeneracy audit; no outcome-driven component replacement',
    'source_axis_refit_authorised':False,'failed_source_rescue_authorised':False,
    'ddo2_fit_authorised_in_this_stage':False,'stage12_authorised':False,'locked_blind_assets_authorised':False,
}
protocol = create_or_verify_seal(PROTOCOL,protocol_payload,'seal_sha256','sealed_utc')

runtime = {'stage':'Stage11F-R','replay_mode':FINAL.is_file(),'target_outcomes_joined':False,
           'ddo2_fitted':False,'stage12_accessed':False,'locked_blind_assets_touched':False}
environment_payload = {
    'stage':'Stage11F-R','python':sys.version,'platform':platform.platform(),
    'numpy':np.__version__,'pandas':pd.__version__,'protocol_seal_sha256':protocol['seal_sha256']
}
if ENVIRONMENT.exists():
    recorded_environment = json.loads(ENVIRONMENT.read_text(encoding='utf-8'))
    assert recorded_environment['protocol_seal_sha256'] == protocol['seal_sha256']
else:
    write_json_once(ENVIRONMENT,environment_payload)
RUNTIME.write_text(json.dumps(runtime,indent=2)+'\n',encoding='utf-8')

print('Stage 11E-R handoff verified:', parent_handoff['handoff_sha256'])
print('Authorised ultrasound edges:', len(edge_roster))
print('Historical eligible edges:', len(historical_library))
print('Stage 9 minimum training edges per grouped fold:', fit_gates['minimum_training_edges_per_grouped_fold'])
print('Protocol seal:', protocol['seal_sha256'])
print('Target outcomes joined before protocol freeze:', runtime['target_outcomes_joined'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 11E-R handoff verified: 98a285c18d7e680a126b78b5dfa70a4883e7c8cf9536aff2a76528adb42f2a5d
Authorised ultrasound edges: 9
Historical eligible edges: 21
Stage 9 minimum training edges per grouped fold: 18
Protocol seal: e4a42c493216d717d96816cf4d0cf455ac95150e6cde18fafdde8c01806234e4
Target outcomes joined before protocol freeze: False


In [11]:
# @title 11F-R-1. Build and freeze label-free components and target predictions for all nine edges
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

split_index = pd.read_csv(PARENT_SPLITS,usecols=['dataset_id','sample_id','group_id','partition','oof_fold'])
assert 'binary_label' not in split_index.columns
assert set(split_index.partition.unique()) == {'development','heldout'}
assert split_index.groupby(['dataset_id','group_id']).partition.nunique().max() == 1

oof_source_labels = pd.read_csv(PARENT_OOF_PREDICTIONS)
heldout_source_reference = pd.read_csv(PARENT_HELDOUT_PREDICTIONS)
assert {'dataset_id','sample_id','group_id','binary_label','decision_score','probability'}.issubset(heldout_source_reference.columns)

def embedding_paths(dataset_id):
    stem = f'{dataset_id}_Canonical_ResNet50_IMAGENET1K_V2_L2'
    root = STAGE11E_ROOT/'01_Frozen_Embeddings'
    return {'embeddings':root/f'{stem}_Embeddings_v0.1.npy','sample_ids':root/f'{stem}_SampleIDs_v0.1.npy'}

def load_partition_embeddings(dataset_id, partition):
    frame = split_index[(split_index.dataset_id==dataset_id)&(split_index.partition==partition)].sort_values('sample_id').reset_index(drop=True)
    paths = embedding_paths(dataset_id)
    row = embedding_manifest[embedding_manifest.dataset_id==dataset_id].iloc[0]
    assert sha_file(paths['embeddings']) == row.embedding_sha256
    assert sha_file(paths['sample_ids']) == row.sample_ids_sha256
    ids = np.load(paths['sample_ids'],allow_pickle=False).astype(str)
    vectors = np.load(paths['embeddings'],mmap_mode='r',allow_pickle=False)
    lookup = {sample_id:index for index,sample_id in enumerate(ids)}
    assert len(lookup)==len(ids) and set(frame.sample_id.astype(str)).issubset(lookup)
    indices = np.asarray([lookup[x] for x in frame.sample_id.astype(str)],dtype=np.int64)
    return frame,np.asarray(vectors[indices],dtype=np.float64)

def load_axis(dataset_id):
    path = STAGE11E_ROOT/'03_Frozen_Source_Axes'/f'{dataset_id}_Frozen_Development_Source_Axis_v0.1.npz'
    row = axis_manifest[axis_manifest.dataset_id==dataset_id].iloc[0]
    assert sha_file(path) == row.axis_npz_sha256 == axis_freeze['axis_sha256_by_dataset'][dataset_id]
    with np.load(path,allow_pickle=False) as z:
        axis = {key:np.asarray(z[key]) for key in z.files}
    assert str(axis['dataset_id']) == dataset_id and bool(axis['axis_frozen_before_heldout_validation']) is True
    return axis

def sigmoid(scores):
    scores = np.clip(np.asarray(scores,dtype=np.float64),-60,60)
    return 1/(1+np.exp(-scores))

def axis_probability(axis,vectors):
    raw = vectors@axis['coefficient_raw'].astype(np.float64)+float(axis['intercept_raw'])
    standardized = (vectors-axis['scaler_mean'].astype(np.float64))/axis['scaler_scale'].astype(np.float64)
    equivalent = standardized@axis['coefficient_standardised'].astype(np.float64)+float(axis['intercept_standardised'])
    maximum_error = float(np.max(np.abs(raw-equivalent)))
    assert maximum_error < AXIS_EQUIVALENCE_TOLERANCE
    return sigmoid(raw),raw,maximum_error

def deterministic_indices(ids,limit,salt):
    ranked = sorted(range(len(ids)),key=lambda i:hashlib.sha256(f'{salt}|{ids[i]}'.encode()).hexdigest())
    return np.asarray(ranked[:min(limit,len(ranked))],dtype=int)

def support_components(source_embeddings,target_embeddings):
    neighbors = min(6,len(source_embeddings))
    assert neighbors >= 2
    model = NearestNeighbors(n_neighbors=neighbors,metric='cosine').fit(source_embeddings)
    source_distances = model.kneighbors(source_embeddings,return_distance=True)[0][:,1:].mean(axis=1)
    target_distances = model.kneighbors(target_embeddings,n_neighbors=min(5,len(source_embeddings)),return_distance=True)[0].mean(axis=1)
    q95,q99 = np.quantile(source_distances,[0.95,0.99])
    median = float(np.median(source_distances))
    return {'support_fraction':float(np.mean(target_distances<=q95)),
            'fraction_beyond_source_q99':float(np.mean(target_distances>q99)),
            'source_knn_q95':float(q95),'source_knn_q99':float(q99),
            'target_mean_knn_distance':float(target_distances.mean()),
            'target_mean_knn_distance_normalised':float(target_distances.mean()/max(median,1e-8))}

def geometry_components(source_table,source_embeddings,target_table,target_embeddings,salt):
    limit = min(128,len(source_table),len(target_table))
    assert limit >= 20
    source_index = deterministic_indices(source_table.sample_id.astype(str).tolist(),limit,salt+'|S')
    target_index = deterministic_indices(target_table.sample_id.astype(str).tolist(),limit,salt+'|T')
    source_sample = source_embeddings[source_index].astype(float)
    target_sample = target_embeddings[target_index].astype(float)
    pooled = np.concatenate([source_sample,target_sample],axis=0)
    domain_labels = np.concatenate([np.zeros(limit,dtype=int),np.ones(limit,dtype=int)])
    components = min(32,pooled.shape[0]-2,pooled.shape[1])
    projected = PCA(n_components=components,svd_solver='randomized',random_state=RANDOM_SEED).fit_transform(pooled)
    classifier = Pipeline([('scale',StandardScaler()),('logistic',LogisticRegression(
        C=1.0,class_weight='balanced',solver='liblinear',random_state=RANDOM_SEED))])
    probability = cross_val_predict(classifier,projected,domain_labels,
        cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_SEED),method='predict_proba')[:,1]
    domain_auc = float(roc_auc_score(domain_labels,probability))
    squared = cdist(source_sample,source_sample,metric='sqeuclidean')
    positive = squared[np.triu_indices_from(squared,k=1)]
    bandwidth = max(float(np.median(positive[positive>0])),1e-8)
    kss = np.exp(-cdist(source_sample,source_sample,metric='sqeuclidean')/bandwidth).mean()
    ktt = np.exp(-cdist(target_sample,target_sample,metric='sqeuclidean')/bandwidth).mean()
    kst = np.exp(-cdist(source_sample,target_sample,metric='sqeuclidean')/bandwidth).mean()
    cost = cdist(source_sample,target_sample,metric='cosine')
    rows,columns = linear_sum_assignment(cost)
    return {'domain_auc':domain_auc,'rbf_mmd2':float(kss+ktt-2*kst),'ot_cosine_cost':float(cost[rows,columns].mean())}

def mixture_components(source_table,target_logits,source_iqr):
    negative = source_table.loc[source_table.binary_label.eq(0),'decision_score'].to_numpy(float)
    positive = source_table.loc[source_table.binary_label.eq(1),'decision_score'].to_numpy(float)
    assert len(negative) and len(positive)
    values = np.concatenate([negative,positive]); best = None
    for prevalence in np.linspace(0,1,101):
        weights = np.concatenate([np.full(len(negative),(1-prevalence)/len(negative)),np.full(len(positive),prevalence/len(positive))])
        distance = wasserstein_distance(np.asarray(target_logits,float),values,
            u_weights=np.full(len(target_logits),1/len(target_logits)),v_weights=weights)
        if best is None or distance < best[0]:
            best = (float(distance),float(prevalence))
    return {'mixture_wasserstein_residual_normalised':float(best[0]/max(source_iqr,1e-8)),
            'unlabeled_mixture_prevalence':best[1]}

def source_calibration_reference(table):
    probability = np.clip(table.probability.to_numpy(float),1e-12,1-1e-12)
    labels = table.binary_label.to_numpy(int)
    error_rate = 1-float(np.mean((probability>=FIXED_THRESHOLD).astype(int)==labels))
    confidence = np.maximum(probability,1-probability)
    return float(np.quantile(confidence,error_rate))

label_free_rows,prediction_rows = [],[]
source_cache = {}
for source in PASSING_SOURCES:
    source_frame,source_vectors = load_partition_embeddings(source,'development')
    source_labels = oof_source_labels[oof_source_labels.dataset_id==source][['sample_id','group_id','binary_label']].copy()
    source_table = source_frame.merge(source_labels,on=['sample_id','group_id'],how='inner',validate='one_to_one')
    assert len(source_table)==len(source_frame) and source_table.binary_label.nunique()==2
    axis = load_axis(source)
    source_probability,source_logits,_ = axis_probability(axis,source_vectors)
    source_table['probability']=source_probability; source_table['decision_score']=source_logits
    source_validation = heldout_source_reference[heldout_source_reference.dataset_id==source].copy()
    source_iqr = float(np.subtract(*np.quantile(source_validation.decision_score.to_numpy(float),[0.75,0.25])))
    atc_threshold = source_calibration_reference(source_validation)
    source_cache[source] = {'axis':axis,'table':source_table,'vectors':source_vectors,'validation':source_validation,
                            'source_iqr':source_iqr,'atc_threshold':atc_threshold}

for edge_index,edge in enumerate(edge_roster.sort_values('edge_id').itertuples(index=False)):
    source,target,edge_id = edge.source_dataset_id,edge.target_dataset_id,edge.edge_id
    assert edge_id == f'{source}__TO__{target}' and source in PASSING_SOURCES and target != source
    cache = source_cache[source]
    target_table,target_vectors = load_partition_embeddings(target,'heldout')
    probabilities,logits,equivalence_error = axis_probability(cache['axis'],target_vectors)
    support = support_components(cache['vectors'],target_vectors)
    geometry = geometry_components(cache['table'],cache['vectors'],target_table,target_vectors,edge_id)
    mixture = mixture_components(cache['table'],logits,cache['source_iqr'])
    confidence = np.maximum(probabilities,1-probabilities)
    entropy = -(probabilities*np.log(np.clip(probabilities,1e-12,1))+(1-probabilities)*np.log(np.clip(1-probabilities,1e-12,1)))
    target_iqr = float(np.subtract(*np.quantile(logits,[0.75,0.25])))
    source_summary = recoverability[recoverability.dataset_id==source].iloc[0]
    label_free_rows.append({
        'edge_id':edge_id,'modality':MODALITY,'task':TASK,'source':source,'target':target,
        'target_units':len(target_table),'source_validation_auc':float(source_summary.heldout_auc),
        'source_validation_auc_ci_lower':float(source_summary.heldout_ci95_lower),'mean_confidence':float(confidence.mean()),
        'mean_entropy_nats':float(entropy.mean()),'atc_estimated_accuracy':float(np.mean(confidence>=cache['atc_threshold'])),
        'atc_threshold_from_source_validation':cache['atc_threshold'],
        'target_to_source_logit_iqr_ratio':float(target_iqr/max(cache['source_iqr'],1e-8)),
        'maximum_axis_equivalence_error':equivalence_error,'is_cross_domain':1,'origin_stage':'Stage11F-R',
        **support,**geometry,**mixture,
    })
    for row,probability,logit in zip(target_table.itertuples(index=False),probabilities,logits):
        prediction_rows.append({'edge_id':edge_id,'modality':MODALITY,'task':TASK,'source':source,'target':target,
            'sample_id':row.sample_id,'group_id':row.group_id,'partition':'heldout',
            'probability':float(probability),'decision_score':float(logit)})

label_free_components = pd.DataFrame(label_free_rows).sort_values('edge_id').reset_index(drop=True)
frozen_predictions = pd.DataFrame(prediction_rows).sort_values(['edge_id','sample_id']).reset_index(drop=True)
assert len(label_free_components)==EXPECTED_NEW_EDGES and label_free_components.edge_id.nunique()==EXPECTED_NEW_EDGES
assert set(label_free_components.edge_id)==set(edge_roster.edge_id)
assert 'binary_label' not in label_free_components.columns and 'binary_label' not in frozen_predictions.columns
assert np.isfinite(label_free_components[frozen_features].to_numpy(float)).all()
assert label_free_components.maximum_axis_equivalence_error.max() < AXIS_EQUIVALENCE_TOLERANCE
write_csv_once(LABEL_FREE_COMPONENTS,label_free_components)
write_csv_once(FROZEN_PREDICTIONS,frozen_predictions)

freeze_payload = {
    'stage':'Stage11F-R','event':'MECHANICAL_DEVELOPMENT_LABEL_ISOLATION_AND_PREDICTION_FREEZE',
    'protocol_seal_sha256':protocol['seal_sha256'],'parent_stage11e_r_handoff_sha256':parent_handoff['handoff_sha256'],
    'authorised_edge_roster_sha256':sha_file(PARENT_EDGE_ROSTER),'frozen_edge_count':len(label_free_components),
    'frozen_edge_ids':label_free_components.edge_id.tolist(),'label_free_components_sha256':sha_file(LABEL_FREE_COMPONENTS),
    'target_predictions_sha256':sha_file(FROZEN_PREDICTIONS),'target_labels_in_frozen_files':False,
    'target_labels_previously_observed_in_stage11e_source_recoverability':True,
    'stage11f_target_outcomes_joined_before_freeze':False,'source_axis_refit':False,'target_model_refit':False,
    'threshold_tuned':False,'feature_replaced_from_transfer_outcome':False,'ddo2_fitted':False,
    'stage12_authorised':False,'locked_blind_assets_touched':False,
}
prediction_freeze = create_or_verify_seal(PREDICTION_FREEZE,freeze_payload,'freeze_sha256','frozen_utc')
print('Frozen label-free edges:',len(label_free_components))
print('Frozen target predictions:',len(frozen_predictions))
print('Prediction freeze:',prediction_freeze['freeze_sha256'])
print('Stage11F target outcomes joined before freeze:',prediction_freeze['stage11f_target_outcomes_joined_before_freeze'])
display(label_free_components[['source','target']+frozen_features])

Frozen label-free edges: 9
Frozen target predictions: 1417
Prediction freeze: f2fecf55a738ac4347b68ab477b54cd17b359825921ec57205148e194dd54358
Stage11F target outcomes joined before freeze: False


,source,target,atc_estimated_accuracy,fraction_beyond_source_q99,mean_entropy_nats,support_fraction,target_to_source_logit_iqr_ratio,unlabeled_mixture_prevalence
0,BUSI_WHU_2025_V3,BUS_BRA_2024,0.806366,0.029178,0.150238,0.835544,1.053264,0.89
1,BUSI_WHU_2025_V3,BUS_UCLM_2025_V3,0.646154,0.123077,0.246899,0.615385,0.849458,0.71
2,BUSI_WHU_2025_V3,RODRIGUES_BUI_2017,1.000000,0.812500,0.010492,0.041667,1.263687,1.00
3,BUS_BRA_2024,BUSI_WHU_2025_V3,0.747312,0.145161,0.167388,0.397849,1.433742,0.47
4,BUS_BRA_2024,BUS_UCLM_2025_V3,0.800000,0.230769,0.131351,0.215385,1.572012,0.50
5,BUS_BRA_2024,RODRIGUES_BUI_2017,0.791667,0.458333,0.113383,0.062500,1.407512,0.20
6,RODRIGUES_BUI_2017,BUSI_WHU_2025_V3,0.892473,1.000000,0.131470,0.000000,1.080855,0.66
7,RODRIGUES_BUI_2017,BUS_BRA_2024,0.872679,0.909814,0.150759,0.013263,0.971263,0.60
8,RODRIGUES_BUI_2017,BUS_UCLM_2025_V3,0.907692,1.000000,0.105842,0.000000,1.782465,0.31


In [12]:
# @title 11F-R-2. Join released outcomes once and evaluate all nine pre-authorised edges
from sklearn.metrics import average_precision_score,brier_score_loss,confusion_matrix,log_loss

prediction_freeze = verify_self(PREDICTION_FREEZE,'freeze_sha256')
assert prediction_freeze['stage11f_target_outcomes_joined_before_freeze'] is False
assert prediction_freeze['target_labels_in_frozen_files'] is False
assert prediction_freeze['label_free_components_sha256'] == sha_file(LABEL_FREE_COMPONENTS)
assert prediction_freeze['target_predictions_sha256'] == sha_file(FROZEN_PREDICTIONS)

target_labels = pd.read_csv(PARENT_SPLITS,usecols=['dataset_id','sample_id','group_id','partition','binary_label'])
target_labels = target_labels[target_labels.partition=='heldout'].copy()
assert set(target_labels.binary_label.astype(int).unique()) == {0,1}

def calibration_error(labels,probabilities,bins=10):
    labels=np.asarray(labels,dtype=int); probabilities=np.asarray(probabilities,dtype=float)
    order=np.argsort(probabilities,kind='mergesort'); bin_ids=np.empty(len(labels),dtype=int)
    bin_ids[order]=np.minimum(np.floor(np.arange(len(labels))*bins/len(labels)).astype(int),bins-1)
    value=0.0
    for bin_id in range(bins):
        mask=bin_ids==bin_id
        if np.any(mask): value += mask.mean()*abs(probabilities[mask].mean()-labels[mask].mean())
    return float(value)

def outcome_metrics(labels,probabilities):
    labels=np.asarray(labels,dtype=int); probabilities=np.clip(np.asarray(probabilities,dtype=float),1e-12,1-1e-12)
    predictions=(probabilities>=FIXED_THRESHOLD).astype(int)
    tn,fp,fn,tp=confusion_matrix(labels,predictions,labels=[0,1]).ravel()
    sensitivity=tp/max(tp+fn,1); specificity=tn/max(tn+fp,1)
    return {'auc':float(roc_auc_score(labels,probabilities)),'average_precision':float(average_precision_score(labels,probabilities)),
            'brier':float(brier_score_loss(labels,probabilities)),'log_loss':float(log_loss(labels,probabilities,labels=[0,1])),
            'ece10':calibration_error(labels,probabilities),'balanced_accuracy':float((sensitivity+specificity)/2),
            'sensitivity':float(sensitivity),'specificity':float(specificity)}

def group_bootstrap_auc(labels,scores,groups,seed):
    labels=np.asarray(labels,dtype=int); scores=np.asarray(scores,dtype=float); groups=np.asarray(groups,dtype=str)
    unique=np.unique(groups); index={group:np.flatnonzero(groups==group) for group in unique}
    rng=np.random.default_rng(seed); values=[]; attempts=0
    while len(values)<N_BOOTSTRAP and attempts<N_BOOTSTRAP*20:
        sampled=rng.choice(unique,size=len(unique),replace=True)
        indices=np.concatenate([index[group] for group in sampled]); attempts+=1
        if np.unique(labels[indices]).size<2: continue
        values.append(roc_auc_score(labels[indices],scores[indices]))
    assert len(values)==N_BOOTSTRAP
    lower,upper=np.quantile(values,[0.025,0.975])
    return float(lower),float(upper),attempts-N_BOOTSTRAP

label_free_components=pd.read_csv(LABEL_FREE_COMPONENTS)
frozen_predictions=pd.read_csv(FROZEN_PREDICTIONS)
edge_rows=[]; evaluated_rows=[]; outcome_rows=[]
for edge_index,edge in enumerate(label_free_components.sort_values('edge_id').itertuples(index=False)):
    pred=frozen_predictions[frozen_predictions.edge_id==edge.edge_id].copy()
    labels=target_labels[target_labels.dataset_id==edge.target][['sample_id','group_id','binary_label']]
    pred=pred.merge(labels,on=['sample_id','group_id'],how='inner',validate='one_to_one')
    assert len(pred)==int(edge.target_units) and pred.binary_label.nunique()==2
    target_metric=outcome_metrics(pred.binary_label,pred.probability)
    seed=RANDOM_SEED+int(hashlib.sha256(edge.edge_id.encode()).hexdigest()[:8],16)%1000000
    lower,upper,rejected=group_bootstrap_auc(pred.binary_label,pred.probability,pred.group_id,seed)
    source_ref=heldout_source_reference[heldout_source_reference.dataset_id==edge.source]
    source_metric=outcome_metrics(source_ref.binary_label,source_ref.probability)
    discrimination_pass=bool(target_metric['auc']>=AUC_MINIMUM and lower>AUC_CI_LOWER_STRICT_MINIMUM)
    calibration_pass=bool(target_metric['ece10']-source_metric['ece10']<=CALIBRATION_DEGRADATION_TOLERANCE and
                          target_metric['brier']-source_metric['brier']<=CALIBRATION_DEGRADATION_TOLERANCE)
    operating_pass=bool(target_metric['balanced_accuracy']>=OPERATING_POINT_BALANCED_ACCURACY_MINIMUM)
    if discrimination_pass and calibration_pass and operating_pass:
        state='DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT_RETAINED'
    elif discrimination_pass and not calibration_pass:
        state='DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED'
    elif discrimination_pass and calibration_pass and not operating_pass:
        state='DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OPERATING_POINT_FAILED'
    else:
        state='DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIONAL_FAILURES'
    row=label_free_components[label_free_components.edge_id==edge.edge_id].iloc[0].to_dict()
    row.update({'source_recoverable':True,'target_auc':target_metric['auc'],'target_auc_ci_lower':lower,
        'target_auc_ci_upper':upper,'source_minus_target_auc':source_metric['auc']-target_metric['auc'],
        'target_average_precision':target_metric['average_precision'],'target_brier':target_metric['brier'],
        'target_ece10':target_metric['ece10'],'calibration_ece_degradation':target_metric['ece10']-source_metric['ece10'],
        'calibration_brier_degradation':target_metric['brier']-source_metric['brier'],
        'target_balanced_accuracy_at_0_5':target_metric['balanced_accuracy'],
        'target_sensitivity_at_0_5':target_metric['sensitivity'],'target_specificity_at_0_5':target_metric['specificity'],
        'discrimination_pass':discrimination_pass,'calibration_pass':calibration_pass,
        'operating_point_pass':operating_pass,'observed_transportability_state':state})
    edge_rows.append(row)
    pred['source']=edge.source; pred['target']=edge.target; pred['modality']=MODALITY; pred['task']=TASK
    evaluated_rows.append(pred)
    outcome_rows.append({'edge_id':edge.edge_id,'source':edge.source,'target':edge.target,
        'target_images':len(pred),'target_groups':pred.group_id.nunique(),
        'target_auc':target_metric['auc'],'target_auc_ci95_lower':lower,'target_auc_ci95_upper':upper,
        'bootstrap_replicates':N_BOOTSTRAP,'rejected_single_class_draws':rejected,
        'discrimination_pass':discrimination_pass,'calibration_pass':calibration_pass,'operating_point_pass':operating_pass})

edge_matrix=pd.DataFrame(edge_rows).sort_values('edge_id').reset_index(drop=True)
evaluated_predictions=pd.concat(evaluated_rows,ignore_index=True).sort_values(['edge_id','sample_id']).reset_index(drop=True)
outcome_summary=pd.DataFrame(outcome_rows).sort_values('edge_id').reset_index(drop=True)
assert len(edge_matrix)==EXPECTED_NEW_EDGES and set(edge_matrix.edge_id)==set(edge_roster.edge_id)
assert edge_matrix[['discrimination_pass','calibration_pass','operating_point_pass']].notna().all().all()
write_csv_once(EDGE_MATRIX,edge_matrix)
write_csv_once(EVALUATED_PREDICTIONS,evaluated_predictions)
write_csv_once(OUTCOME_SUMMARY,outcome_summary)
runtime['target_outcomes_joined']=True
RUNTIME.write_text(json.dumps(runtime,indent=2)+'\n',encoding='utf-8')
display(outcome_summary)
print('All authorised edge outcomes evaluated:',len(edge_matrix))
print('Target labels joined only after prediction freeze:',prediction_freeze['freeze_sha256'])

,edge_id,source,target,target_images,target_groups,target_auc,target_auc_ci95_lower,target_auc_ci95_upper,bootstrap_replicates,rejected_single_class_draws,discrimination_pass,calibration_pass,operating_point_pass
0,BUSI_WHU_2025_V3__TO__BUS_BRA_2024,BUSI_WHU_2025_V3,BUS_BRA_2024,377,212,0.548891,0.478470,0.618088,2000,0,False,False,False
1,BUSI_WHU_2025_V3__TO__BUS_UCLM_2025_V3,BUSI_WHU_2025_V3,BUS_UCLM_2025_V3,65,7,0.549683,0.212047,0.800010,2000,48,False,False,False
2,BUSI_WHU_2025_V3__TO__RODRIGUES_BUI_2017,BUSI_WHU_2025_V3,RODRIGUES_BUI_2017,48,48,0.183303,0.066406,0.328512,2000,0,False,False,False
3,BUS_BRA_2024__TO__BUSI_WHU_2025_V3,BUS_BRA_2024,BUSI_WHU_2025_V3,186,163,0.567447,0.482976,0.654435,2000,0,False,False,False
4,BUS_BRA_2024__TO__BUS_UCLM_2025_V3,BUS_BRA_2024,BUS_UCLM_2025_V3,65,7,0.694503,0.466485,0.788038,2000,42,False,True,False
5,BUS_BRA_2024__TO__RODRIGUES_BUI_2017,BUS_BRA_2024,RODRIGUES_BUI_2017,48,48,0.829401,0.694431,0.939085,2000,0,True,False,False
6,RODRIGUES_BUI_2017__TO__BUSI_WHU_2025_V3,RODRIGUES_BUI_2017,BUSI_WHU_2025_V3,186,163,0.475567,0.392714,0.562843,2000,0,False,False,False
7,RODRIGUES_BUI_2017__TO__BUS_BRA_2024,RODRIGUES_BUI_2017,BUS_BRA_2024,377,212,0.530698,0.457443,0.601980,2000,0,False,False,False
8,RODRIGUES_BUI_2017__TO__BUS_UCLM_2025_V3,RODRIGUES_BUI_2017,BUS_UCLM_2025_V3,65,7,0.451374,0.207265,0.690713,2000,39,False,False,False


All authorised edge outcomes evaluated: 9
Target labels joined only after prediction freeze: f2fecf55a738ac4347b68ab477b54cd17b359825921ec57205148e194dd54358


In [13]:
# @title 11F-R-3. Append the nine-edge matrix and audit overlap, duplicates, incidence, and dynamic range
historical_library=pd.read_csv(STAGE8B_LIBRARY)
edge_matrix=pd.read_csv(EDGE_MATRIX)
assert len(historical_library)==EXPECTED_PARENT_EDGES and len(edge_matrix)==EXPECTED_NEW_EDGES
assert set(historical_library.edge_id).isdisjoint(set(edge_matrix.edge_id))

common_columns=sorted(set(historical_library.columns)|set(edge_matrix.columns))
historical_part=historical_library.reindex(columns=common_columns).copy()
new_part=edge_matrix.reindex(columns=common_columns).copy()

# Preserve the sealed Stage 8B schema through concatenation. Without this,
# pandas can promote numeric columns to object when the new table carries a
# different inferred dtype; object-valued floats then bypass canonical
# float_format and acquire representation-only changes on CSV round-trip.
for column in historical_library.columns:
    dtype=historical_library[column].dtype
    if pd.api.types.is_bool_dtype(dtype):
        new_part[column]=new_part[column].astype(bool)
    elif pd.api.types.is_numeric_dtype(dtype):
        new_part[column]=pd.to_numeric(new_part[column],errors='raise').astype(dtype)
    else:
        new_part[column]=new_part[column].astype(object)

expanded_library=pd.concat([historical_part,new_part],ignore_index=True)
assert len(expanded_library)==EXPECTED_TOTAL_IF_COMPLETE and expanded_library.edge_id.nunique()==EXPECTED_TOTAL_IF_COMPLETE
assert not expanded_library.duplicated(['source','target']).any()

historical_datasets=set(historical_library.source.astype(str))|set(historical_library.target.astype(str))
new_datasets=set(edge_matrix.source.astype(str))|set(edge_matrix.target.astype(str))
overlap=sorted(historical_datasets&new_datasets)

model_and_outcome_columns=frozen_features+['discrimination_pass','calibration_pass','operating_point_pass']
exact_rows=int(expanded_library.duplicated(model_and_outcome_columns,keep=False).sum())
duplicate_rows=[
    {'check':'authorised_edge_ids_complete','observed':len(set(edge_matrix.edge_id)&set(edge_roster.edge_id)),'expected':EXPECTED_NEW_EDGES,'passed':set(edge_matrix.edge_id)==set(edge_roster.edge_id)},
    {'check':'expanded_edge_ids_unique','observed':expanded_library.edge_id.nunique(),'expected':len(expanded_library),'passed':expanded_library.edge_id.is_unique},
    {'check':'source_target_pairs_unique','observed':int(expanded_library.duplicated(['source','target']).sum()),'expected':0,'passed':not expanded_library.duplicated(['source','target']).any()},
    {'check':'historical_new_dataset_identity_overlap','observed':len(overlap),'expected':0,'passed':len(overlap)==0},
    {'check':'exact_feature_plus_outcome_duplicate_rows','observed':exact_rows,'expected':0,'passed':exact_rows==0},
]
duplicate_audit=pd.DataFrame(duplicate_rows)

range_columns=frozen_features+['target_auc','source_minus_target_auc','target_ece10','calibration_ece_degradation',
    'calibration_brier_degradation','target_balanced_accuracy_at_0_5']
range_rows=[]
for column in range_columns:
    values=pd.to_numeric(expanded_library[column],errors='coerce')
    range_rows.append({'variable':column,'role':'FROZEN_MODEL_INPUT' if column in frozen_features else 'OUTCOME_AUDIT',
        'nonmissing':int(values.notna().sum()),'unique_values':int(values.nunique(dropna=True)),
        'minimum':float(values.min()),'maximum':float(values.max()),'range':float(values.max()-values.min()),
        'nondegenerate':bool(values.notna().all() and values.nunique(dropna=True)>1 and float(values.max()-values.min())>1e-12)})
dynamic_range_audit=pd.DataFrame(range_rows)

all_datasets=sorted(set(expanded_library.source.astype(str))|set(expanded_library.target.astype(str)))
incidence_rows=[]
for dataset in all_datasets:
    incoming=int(expanded_library.target.eq(dataset).sum()); outgoing=int(expanded_library.source.eq(dataset).sum())
    incidence_rows.append({'dataset':dataset,'incoming_edges':incoming,'outgoing_edges':outgoing,
                           'incident_edges':incoming+outgoing,'modalities':'|'.join(sorted(expanded_library.loc[
                               expanded_library.source.eq(dataset)|expanded_library.target.eq(dataset),'modality'].astype(str).unique()))})
dataset_incidence=pd.DataFrame(incidence_rows)

write_csv_once(EXPANDED_LIBRARY,expanded_library)
write_csv_once(DUPLICATE_AUDIT,duplicate_audit)
write_csv_once(DYNAMIC_RANGE_AUDIT,dynamic_range_audit)
write_csv_once(DATASET_INCIDENCE,dataset_incidence)
library_payload={'stage':'Stage11F-R','event':'FOUR_MODALITY_DEVELOPMENT_EDGE_LIBRARY_FREEZE',
    'protocol_seal_sha256':protocol['seal_sha256'],'parent_library_sha256':sha_file(STAGE8B_LIBRARY),
    'ultrasound_edge_matrix_sha256':sha_file(EDGE_MATRIX),'expanded_library_sha256':sha_file(EXPANDED_LIBRARY),
    'eligible_edges':len(expanded_library),'modalities':int(expanded_library.modality.nunique()),
    'unique_datasets':len(all_datasets),'authorised_ultrasound_edges_complete':len(edge_matrix)==EXPECTED_NEW_EDGES,
    'historical_new_dataset_identity_overlap':overlap,'ddo2_fitted':False,'stage12_authorised':False,
    'locked_blind_assets_touched':False}
library_freeze=create_or_verify_seal(LIBRARY_FREEZE,library_payload,'freeze_sha256','frozen_utc')
display(duplicate_audit)
display(dynamic_range_audit)
print('Expanded library:',len(expanded_library),'edges /',expanded_library.modality.nunique(),'modalities /',len(all_datasets),'datasets')
print('Expanded library freeze:',library_freeze['freeze_sha256'])

,check,observed,expected,passed
0,authorised_edge_ids_complete,9,9,True
1,expanded_edge_ids_unique,30,30,True
2,source_target_pairs_unique,0,0,True
3,historical_new_dataset_identity_overlap,0,0,True
4,exact_feature_plus_outcome_duplicate_rows,0,0,True


,variable,role,nonmissing,unique_values,minimum,maximum,range,nondegenerate
0,atc_estimated_accuracy,FROZEN_MODEL_INPUT,30,30,0.465000,1.000000,0.535000,True
1,fraction_beyond_source_q99,FROZEN_MODEL_INPUT,30,29,0.000000,1.000000,1.000000,True
2,mean_entropy_nats,FROZEN_MODEL_INPUT,30,30,0.010492,0.380123,0.369631,True
3,support_fraction,FROZEN_MODEL_INPUT,30,26,0.000000,0.933735,0.933735,True
4,target_to_source_logit_iqr_ratio,FROZEN_MODEL_INPUT,30,30,0.412186,1.782465,1.370279,True
5,unlabeled_mixture_prevalence,FROZEN_MODEL_INPUT,30,25,0.050000,1.000000,0.950000,True
6,target_auc,OUTCOME_AUDIT,30,30,0.183303,0.884040,0.700737,True
7,source_minus_target_auc,OUTCOME_AUDIT,30,30,-0.107415,0.618217,0.725632,True
8,target_ece10,OUTCOME_AUDIT,30,30,0.117744,0.640584,0.522840,True
9,calibration_ece_degradation,OUTCOME_AUDIT,30,30,-0.099755,0.452900,0.552656,True


Expanded library: 30 edges / 4 modalities / 14 datasets
Expanded library freeze: e9e6fd5a5eeaf62be02abee5dac6234812d6745bac13db34aab0ea93987761c3


In [14]:
# @title 11F-R-4. Rebuild grouped folds and apply the frozen Stage 9 DDO-2 fit gates
expanded_library=pd.read_csv(EXPANDED_LIBRARY)
fit_gates=stage9_model_spec['fit_gates']
axis_columns={'discrimination':'discrimination_failure','calibration':'calibration_failure','operating_point':'operating_point_failure'}
for axis,failure_column in axis_columns.items():
    pass_column=f'{axis}_pass'
    expanded_library[failure_column]=1-expanded_library[pass_column].astype(bool).astype(int)

all_datasets=sorted(set(expanded_library.source.astype(str))|set(expanded_library.target.astype(str)))
fold_rows=[]
def fold_row(scheme,held_out_group,test_mask):
    train=expanded_library.loc[~test_mask]; test=expanded_library.loc[test_mask]
    row={'scheme':scheme,'held_out_group':held_out_group,'n_train_edges':len(train),'n_test_edges':len(test),
         'group_leakage_detected':False}
    if scheme=='LEAVE_ONE_DATASET_OUT':
        row['group_leakage_detected']=bool(train.source.eq(held_out_group).any() or train.target.eq(held_out_group).any())
    else:
        row['group_leakage_detected']=bool(train.modality.eq(held_out_group).any())
    for axis,failure_column in axis_columns.items():
        row[f'train_{axis}_failures']=int(train[failure_column].sum()); row[f'train_{axis}_passes']=int(len(train)-train[failure_column].sum())
        row[f'test_{axis}_failures']=int(test[failure_column].sum()); row[f'test_{axis}_passes']=int(len(test)-test[failure_column].sum())
    row['fold_meets_minimum_training_edges']=bool(len(train)>=fit_gates['minimum_training_edges_per_grouped_fold'])
    return row

for dataset in all_datasets:
    fold_rows.append(fold_row('LEAVE_ONE_DATASET_OUT',dataset,expanded_library.source.eq(dataset)|expanded_library.target.eq(dataset)))
for modality in sorted(expanded_library.modality.astype(str).unique()):
    fold_rows.append(fold_row('LEAVE_ONE_MODALITY_OUT',modality,expanded_library.modality.eq(modality)))
fold_registry=pd.DataFrame(fold_rows)
assert not fold_registry.group_leakage_detected.any()
write_csv_once(FOLD_REGISTRY,fold_registry)

eligibility_rows=[]
def gate(scope,gate_name,observed,required,passed):
    eligibility_rows.append({'scope':scope,'gate':gate_name,'observed':observed,'required':required,
        'passed':bool(passed),'consequence_if_failed':'DDO2_COEFFICIENT_FIT_NOT_AUTHORISED'})

gate('GLOBAL','total_eligible_edges',len(expanded_library),fit_gates['minimum_total_eligible_edges'],len(expanded_library)>=fit_gates['minimum_total_eligible_edges'])
gate('GLOBAL','modalities',expanded_library.modality.nunique(),fit_gates['minimum_modalities'],expanded_library.modality.nunique()>=fit_gates['minimum_modalities'])
gate('GLOBAL','unique_datasets',len(all_datasets),fit_gates['minimum_unique_datasets'],len(all_datasets)>=fit_gates['minimum_unique_datasets'])
gate('GLOBAL','all_grouped_folds_minimum_training_edges',int(fold_registry.fold_meets_minimum_training_edges.sum()),len(fold_registry),fold_registry.fold_meets_minimum_training_edges.all())
for axis,failure_column in axis_columns.items():
    failures=int(expanded_library[failure_column].sum()); passes=int(len(expanded_library)-failures)
    failure_modalities=int(expanded_library.loc[expanded_library[failure_column].eq(1),'modality'].nunique())
    pass_modalities=int(expanded_library.loc[expanded_library[failure_column].eq(0),'modality'].nunique())
    gate(axis,'minimum_failures',failures,fit_gates['minimum_failures_per_axis'],failures>=fit_gates['minimum_failures_per_axis'])
    gate(axis,'minimum_passes',passes,fit_gates['minimum_passes_per_axis'],passes>=fit_gates['minimum_passes_per_axis'])
    gate(axis,'failure_class_modalities',failure_modalities,fit_gates['minimum_modalities_per_axis_class'],failure_modalities>=fit_gates['minimum_modalities_per_axis_class'])
    gate(axis,'pass_class_modalities',pass_modalities,fit_gates['minimum_modalities_per_axis_class'],pass_modalities>=fit_gates['minimum_modalities_per_axis_class'])
fit_eligibility=pd.DataFrame(eligibility_rows)
write_csv_once(FIT_ELIGIBILITY,fit_eligibility)

duplicate_audit=pd.read_csv(DUPLICATE_AUDIT)
dynamic_range_audit=pd.read_csv(DYNAMIC_RANGE_AUDIT)
assembly_checks={
    'all_nine_authorised_edges_present':bool(len(pd.read_csv(EDGE_MATRIX))==EXPECTED_NEW_EDGES and set(pd.read_csv(EDGE_MATRIX).edge_id)==set(edge_roster.edge_id)),
    'duplicate_and_identity_overlap_checks_pass':bool(duplicate_audit.passed.astype(bool).all()),
    'frozen_model_inputs_and_outcomes_nondegenerate':bool(dynamic_range_audit.nondegenerate.astype(bool).all()),
    'all_stage9_fit_gates_pass':bool(fit_eligibility.passed.astype(bool).all()),
    'all_grouped_folds_meet_18_edge_threshold':bool(fold_registry.fold_meets_minimum_training_edges.astype(bool).all()),
}
FIT_READY=bool(all(assembly_checks.values()))
decision=('SEAL_STAGE11F_R_AUTHORISE_STAGE11G_R_DEVELOPMENT_ONLY_DDO2_FIT_AND_STRONG_BASELINE_COMPARISON_KEEP_STAGE12_AND_BLIND_PROHIBITED'
          if FIT_READY else
          'SEAL_STAGE11F_R_HOLD_DDO2_FIT_READINESS_KEEP_STAGE12_AND_BLIND_PROHIBITED')
failed_fit_gates = json.loads(fit_eligibility.loc[~fit_eligibility.passed.astype(bool),
    ['scope','gate','observed','required']].to_json(orient='records'))
decision_payload={'stage':'Stage11F-R','decision':decision,'protocol_seal_sha256':protocol['seal_sha256'],
    'prediction_freeze_sha256':prediction_freeze['freeze_sha256'],'expanded_library_freeze_sha256':library_freeze['freeze_sha256'],
    'expanded_library_sha256':sha_file(EXPANDED_LIBRARY),'fit_eligibility_sha256':sha_file(FIT_ELIGIBILITY),
    'fold_registry_sha256':sha_file(FOLD_REGISTRY),'assembly_checks':assembly_checks,'fit_ready':FIT_READY,
    'failed_fit_gates':failed_fit_gates,
    'stage11g_r_development_only_fit_authorised':FIT_READY,'ddo2_fitted':False,'stage12_authorised':False,
    'locked_blind_assets_touched':False}
assembly_decision=create_or_verify_seal(ASSEMBLY_DECISION,decision_payload,'decision_sha256','sealed_utc')
display(fit_eligibility)
display(fold_registry[['scheme','held_out_group','n_train_edges','n_test_edges','fold_meets_minimum_training_edges']])
print('Fit ready:',FIT_READY)
print('Decision:',decision)
print('Decision seal:',assembly_decision['decision_sha256'])

,scope,gate,observed,required,passed,consequence_if_failed
0,GLOBAL,total_eligible_edges,30,30,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
1,GLOBAL,modalities,4,4,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
2,GLOBAL,unique_datasets,14,12,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
3,GLOBAL,all_grouped_folds_minimum_training_edges,18,18,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
4,discrimination,minimum_failures,19,6,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
5,discrimination,minimum_passes,11,6,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
6,discrimination,failure_class_modalities,4,2,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
7,discrimination,pass_class_modalities,4,2,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
8,calibration,minimum_failures,26,6,True,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED
9,calibration,minimum_passes,4,6,False,DDO2_COEFFICIENT_FIT_NOT_AUTHORISED


,scheme,held_out_group,n_train_edges,n_test_edges,fold_meets_minimum_training_edges
0,LEAVE_ONE_DATASET_OUT,APTOS_2019,25,5,True
1,LEAVE_ONE_DATASET_OUT,BUSI_WHU_2025_V3,25,5,True
2,LEAVE_ONE_DATASET_OUT,BUS_BRA_2024,25,5,True
3,LEAVE_ONE_DATASET_OUT,BUS_UCLM_2025_V3,27,3,True
4,LEAVE_ONE_DATASET_OUT,DeepDRiD,25,5,True
5,LEAVE_ONE_DATASET_OUT,EyePACS_2015,27,3,True
6,LEAVE_ONE_DATASET_OUT,HAM10000,26,4,True
7,LEAVE_ONE_DATASET_OUT,IDRiD,25,5,True
8,LEAVE_ONE_DATASET_OUT,ISIC_MSK1,26,4,True
9,LEAVE_ONE_DATASET_OUT,ISIC_UDA1,26,4,True


Fit ready: False
Decision: SEAL_STAGE11F_R_HOLD_DDO2_FIT_READINESS_KEEP_STAGE12_AND_BLIND_PROHIBITED
Decision seal: 4b33aef11f93de4e2f7b7ed7b42ea52e0e66f6dbe5273a247dfc24ff673b7d80


In [15]:
# @title 11F-R-5. Run independent lineage, chronology, leakage, and blind-firewall checks
checks=[]
def check(name,passed,evidence):
    checks.append({'check':name,'passed':bool(passed),'evidence':str(evidence)[:2000]})

prediction_freeze=verify_self(PREDICTION_FREEZE,'freeze_sha256')
library_freeze=verify_self(LIBRARY_FREEZE,'freeze_sha256')
assembly_decision=verify_self(ASSEMBLY_DECISION,'decision_sha256')
label_free_components=pd.read_csv(LABEL_FREE_COMPONENTS)
frozen_predictions=pd.read_csv(FROZEN_PREDICTIONS)
edge_matrix=pd.read_csv(EDGE_MATRIX)
expanded_library=pd.read_csv(EXPANDED_LIBRARY)
fit_eligibility=pd.read_csv(FIT_ELIGIBILITY)
fold_registry=pd.read_csv(FOLD_REGISTRY)

check('stage11e_handoff_exact',parent_handoff['handoff_sha256']==EXPECTED_STAGE11E_HANDOFF,parent_handoff['handoff_sha256'])
check('stage11e_final_exact',parent_final['final_record_sha256']==EXPECTED_STAGE11E_FINAL,parent_final['final_record_sha256'])
check('stage8b_library_exact',sha_file(STAGE8B_LIBRARY)==EXPECTED_STAGE8B_LIBRARY,sha_file(STAGE8B_LIBRARY))
check('stage9_model_spec_exact',stage9_model_spec['specification_sha256']==EXPECTED_STAGE9_MODEL_SPEC,stage9_model_spec['specification_sha256'])
check('notebook_source_matches_protocol',notebook_source_sha256(NOTEBOOK_PATH)==protocol['notebook_source_sha256'],notebook_source_sha256(NOTEBOOK_PATH))
check('authorised_edge_roster_complete',set(edge_matrix.edge_id)==set(edge_roster.edge_id),len(edge_matrix))
check('all_edge_targets_use_heldout_partition',set(frozen_predictions.partition)=={'heldout'},sorted(frozen_predictions.partition.unique()))
check('label_free_files_exclude_binary_label','binary_label' not in label_free_components.columns and 'binary_label' not in frozen_predictions.columns,list(frozen_predictions.columns))
check('target_outcome_join_after_freeze',prediction_freeze['stage11f_target_outcomes_joined_before_freeze'] is False,prediction_freeze['freeze_sha256'])
check('parent_target_labels_already_observed_transparently',prediction_freeze['target_labels_previously_observed_in_stage11e_source_recoverability'] is True,'development-only mechanical isolation')
check('frozen_axis_equivalence',label_free_components.maximum_axis_equivalence_error.max()<AXIS_EQUIVALENCE_TOLERANCE,label_free_components.maximum_axis_equivalence_error.max())
historical_candidate=expanded_library.iloc[:EXPECTED_PARENT_EDGES].reindex(columns=historical_library.columns).reset_index(drop=True)
historical_reference=historical_library.reset_index(drop=True)

def frames_semantically_identical(candidate,reference):
    if list(candidate.columns)!=list(reference.columns) or len(candidate)!=len(reference):
        return False
    if candidate.edge_id.astype(str).tolist()!=reference.edge_id.astype(str).tolist():
        return False
    try:
        pd.testing.assert_frame_equal(candidate,reference,check_dtype=False,check_exact=False,rtol=1e-12,atol=0.0)
    except AssertionError:
        return False
    return True

historical_rows_preserved=frames_semantically_identical(historical_candidate,historical_reference)
check('historical_rows_preserved',historical_rows_preserved,
      f'{EXPECTED_PARENT_EDGES} rows / {len(historical_library.columns)} columns / schema-aware semantic identity')
check('new_rows_exactly_nine',len(edge_matrix)==EXPECTED_NEW_EDGES,len(edge_matrix))
check('expanded_rows_exactly_thirty',len(expanded_library)==EXPECTED_TOTAL_IF_COMPLETE,len(expanded_library))
check('grouped_fold_leakage_absent',not fold_registry.group_leakage_detected.astype(bool).any(),int(fold_registry.group_leakage_detected.astype(bool).sum()))
check('stage9_18_edge_fold_threshold_used',stage9_model_spec['fit_gates']['minimum_training_edges_per_grouped_fold']==18,stage9_model_spec['fit_gates'])
check('fit_decision_matches_all_gates',assembly_decision['fit_ready']==bool(fit_eligibility.passed.astype(bool).all() and all(v for k,v in assembly_decision['assembly_checks'].items() if k!='all_stage9_fit_gates_pass')),assembly_decision['fit_ready'])
check('ddo2_not_fitted',assembly_decision['ddo2_fitted'] is False,assembly_decision['ddo2_fitted'])
check('stage12_not_authorised',assembly_decision['stage12_authorised'] is False,assembly_decision['stage12_authorised'])
configured_paths='|'.join(str(x) for x in required+[ROOT])
check('locked_asset_markers_absent_from_configured_paths',not any(marker.lower() in configured_paths.lower() for marker in LOCKED_ASSET_MARKERS),configured_paths)
check('locked_blind_assets_not_touched',runtime['locked_blind_assets_touched'] is False,runtime['locked_blind_assets_touched'])

firewall=pd.DataFrame(checks)
write_csv_once(FIREWALL,firewall)
assert firewall.passed.all(),firewall.loc[~firewall.passed].to_dict('records')
display(firewall)
print('Independent checks passed:',int(firewall.passed.sum()),'/',len(firewall))

,check,passed,evidence
0,stage11e_handoff_exact,True,98a285c18d7e680a126b78b5dfa70a4883e7c8cf9536af...
1,stage11e_final_exact,True,24d41b7a3a30b1548a81ab1a99d909eb29bf9189fef497...
2,stage8b_library_exact,True,60ee87ec253c34d2e3b50f6f1a40318aa1a14c0f0e5bf1...
3,stage9_model_spec_exact,True,e2a0d8f65143e0b9d58d740595a39b803f1684481c4f8a...
4,notebook_source_matches_protocol,True,c238efca7349cf3c73d300357fdc3008dfee657d43f824...
5,authorised_edge_roster_complete,True,9
6,all_edge_targets_use_heldout_partition,True,['heldout']
7,label_free_files_exclude_binary_label,True,"['edge_id', 'modality', 'task', 'source', 'tar..."
8,target_outcome_join_after_freeze,True,f2fecf55a738ac4347b68ab477b54cd17b359825921ec5...
9,parent_target_labels_already_observed_transpar...,True,development-only mechanical isolation


Independent checks passed: 21 / 21


In [16]:
# @title 11F-R-6. Seal report, integrity manifest, final record, and next-stage handoff
firewall=pd.read_csv(FIREWALL)
edge_matrix=pd.read_csv(EDGE_MATRIX)
expanded_library=pd.read_csv(EXPANDED_LIBRARY)
fit_eligibility=pd.read_csv(FIT_ELIGIBILITY)
fold_registry=pd.read_csv(FOLD_REGISTRY)
assembly_decision=verify_self(ASSEMBLY_DECISION,'decision_sha256')
failed=fit_eligibility.loc[~fit_eligibility.passed.astype(bool),['scope','gate','observed','required']]

report=f"""# Stage 11F-R report

## Answer first

**Decision:** `{assembly_decision['decision']}`

Stage 11F-R assembled and evaluated all {len(edge_matrix)} pre-authorised breast-ultrasound development edges and produced a {len(expanded_library)}-edge, {expanded_library.modality.nunique()}-modality development library. DDO-2 coefficients were not fitted. Stage 12 and locked-blind assets remained prohibited.

## Chronology

- Ultrasound target labels were already observed in Stage 11E-R for source recoverability.
- Stage 11F-R nevertheless excluded target labels from its component and prediction files and sealed those files before joining outcomes in this notebook.
- This is a development label-isolation control, not a new prospective blind unseal.

## New ultrasound outcome states

{markdown_table(edge_matrix[['source','target','target_auc','target_auc_ci_lower','target_ece10','target_balanced_accuracy_at_0_5','discrimination_pass','calibration_pass','operating_point_pass']])}

## Frozen Stage 9 fit-readiness audit

{markdown_table(fit_eligibility)}

The grouped-fold threshold was read directly from the frozen Stage 9 model specification: at least {stage9_model_spec['fit_gates']['minimum_training_edges_per_grouped_fold']} training edges in every leave-one-dataset-out and leave-one-modality-out fold.

## Boundary

- Stage 11G-R development-only fitting is authorised: **{assembly_decision['stage11g_r_development_only_fit_authorised']}**.
- DDO-2 fitted here: **False**.
- Stage 12 authorised: **False**.
- Locked-blind assets touched: **False**.
"""
write_text_once(REPORT,report)

tracked=[]
for path in sorted(ROOT.rglob('*')):
    if not path.is_file() or path in {RUNTIME,OUTPUT_MANIFEST,FINAL}:
        continue
    tracked.append({'relative_path':str(path.relative_to(ROOT)),'size_bytes':path.stat().st_size,'sha256':sha_file(path)})
output_manifest=pd.DataFrame(tracked)
write_csv_once(OUTPUT_MANIFEST,output_manifest)

final_payload={'stage':'Stage11F-R','decision':assembly_decision['decision'],'protocol_seal_sha256':protocol['seal_sha256'],
    'parent_stage11e_r_handoff_sha256':parent_handoff['handoff_sha256'],'prediction_freeze_sha256':prediction_freeze['freeze_sha256'],
    'ultrasound_edge_matrix_sha256':sha_file(EDGE_MATRIX),'expanded_library_sha256':sha_file(EXPANDED_LIBRARY),
    'expanded_library_freeze_sha256':library_freeze['freeze_sha256'],'fit_eligibility_sha256':sha_file(FIT_ELIGIBILITY),
    'assembly_decision_sha256':assembly_decision['decision_sha256'],'firewall_sha256':sha_file(FIREWALL),
    'output_integrity_manifest_sha256':sha_file(OUTPUT_MANIFEST),'ultrasound_edges_evaluated':len(edge_matrix),
    'eligible_edges_total':len(expanded_library),'modalities_total':int(expanded_library.modality.nunique()),
    'unique_datasets_total':len(set(expanded_library.source.astype(str))|set(expanded_library.target.astype(str))),
    'grouped_folds_total':len(fold_registry),'grouped_folds_meeting_18_edge_threshold':int(fold_registry.fold_meets_minimum_training_edges.astype(bool).sum()),
    'failed_fit_gate_count':len(failed),'stage11g_r_development_only_fit_authorised':assembly_decision['stage11g_r_development_only_fit_authorised'],
    'ddo2_fitted':False,'stage12_authorised':False,'locked_blind_assets_touched':False,
    'next_step':('BUILD_STAGE11G_R_DEVELOPMENT_ONLY_DDO2_FIT_AND_STRONG_BASELINE_COMPARISON' if assembly_decision['stage11g_r_development_only_fit_authorised']
                 else 'HOLD_DDO2_FIT_AND_REVIEW_FAILED_STAGE11F_R_READINESS_GATES')}
final=create_or_verify_seal(FINAL,final_payload,'final_record_sha256','completed_utc')
runtime['completed']=True; runtime['final_record_sha256']=final['final_record_sha256']
RUNTIME.write_text(json.dumps(runtime,indent=2)+'\n',encoding='utf-8')

print('================ STAGE 11F-R COMPLETE ================')
print('Ultrasound edges evaluated / authorised:',len(edge_matrix),'/',EXPECTED_NEW_EDGES)
print('Expanded eligible edges:',len(expanded_library))
print('Modalities / unique datasets:',expanded_library.modality.nunique(),'/',final['unique_datasets_total'])
print('Grouped folds meeting 18-edge threshold:',final['grouped_folds_meeting_18_edge_threshold'],'/',final['grouped_folds_total'])
print('Failed fit gates:',final['failed_fit_gate_count'])
print('Stage11G-R development-only fit authorised:',final['stage11g_r_development_only_fit_authorised'])
print('Decision:',final['decision'])
print('Protocol seal:',final['protocol_seal_sha256'])
print('Prediction freeze hash:',final['prediction_freeze_sha256'])
print('Expanded library hash:',final['expanded_library_sha256'])
print('Fit eligibility hash:',final['fit_eligibility_sha256'])
print('Final record hash:',final['final_record_sha256'])
print('Next step:',final['next_step'])

================ STAGE 11F-R COMPLETE ================
Ultrasound edges evaluated / authorised: 9 / 9
Expanded eligible edges: 30
Modalities / unique datasets: 4 / 14
Grouped folds meeting 18-edge threshold: 18 / 18
Failed fit gates: 3
Stage11G-R development-only fit authorised: False
Decision: SEAL_STAGE11F_R_HOLD_DDO2_FIT_READINESS_KEEP_STAGE12_AND_BLIND_PROHIBITED
Protocol seal: e4a42c493216d717d96816cf4d0cf455ac95150e6cde18fafdde8c01806234e4
Prediction freeze hash: f2fecf55a738ac4347b68ab477b54cd17b359825921ec57205148e194dd54358
Expanded library hash: f074467614321e4162a3aa4ad982b62edfa362d6238edeb2f69be6d4d50f09a9
Fit eligibility hash: 5d1e5d045a3dff69f7888e1d118ccfff79ec9760c9dd92cb44fbf3d69f3eece0
Final record hash: bfe88b0b57d9efc90a123995d606648e9074eecb17d951087caaf4a889514423
Next step: HOLD_DDO2_FIT_AND_REVIEW_FAILED_STAGE11F_R_READINESS_GATES
